# 11장. 임베딩과 어텐션

영화평을 토큰으로 나누고, 임베딩의 평균으로 긍정과 부정을 분류한다. 어텐션을 넘파이로 계산하고, 답을 아는 과제에서 어텐션이 맞는 곳을 보는지 확인한다.

**준비:** 네이버 영화평 자료(NSMC)의 학습용 파일 `ratings_train.txt`를 왼쪽 파일 창에 올린 뒤 위에서부터 차례로 실행한다. 받는 곳은 https://github.com/e9t/nsmc 다. 시험용 파일 `ratings_test.txt`는 16장에서 쓴다. 끝까지 실행하는 데 10분쯤 걸린다.

## 11.1 문장을 토큰으로 나누기: 글자, 어절, 부분 단어

In [ ]:
import time
from collections import Counter
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn

torch.set_num_threads(2)
raw = pd.read_csv("ratings_train.txt", sep="\t")
print(raw.shape, list(raw.columns), int(raw["document"].isna().sum()))
print(raw["label"].value_counts().to_dict())
raw = raw.dropna(subset=["document"]).reset_index(drop=True)

order = np.random.default_rng(0).permutation(len(raw))
train, valid = raw.iloc[order[:30000]], raw.iloc[order[-10000:]]
texts_tr, texts_va = train["document"].tolist(), valid["document"].tolist()
y_tr = torch.tensor(train["label"].to_numpy())
y_va = torch.tensor(valid["label"].to_numpy())
print(len(texts_tr), len(texts_va), len(raw) - 40000,
      round(y_tr.float().mean().item(), 4), round(y_va.float().mean().item(), 4))
for row in [10, 54, 87, 83]:
    print(y_tr[row].item(), texts_tr[row])
sizes = np.array([len(text) for text in texts_tr])
print(int(np.median(sizes)), round(sizes.mean(), 1), sizes.max(),
      round((sizes > 64).mean() * 100, 1))

In [ ]:
PAD, UNKNOWN = 0, 1                     # 0번은 패딩, 1번은 어휘에 없는 토큰
by_char, by_word = list, str.split      # 글자로 나누기, 띄어쓰기로 나누기

def make_vocab(texts, split, least):
    """least번 이상 나온 토큰에 많이 나온 순서로 2번부터 번호를 매긴다."""
    counts = Counter(token for text in texts for token in split(text))
    kept = [token for token, n in counts.most_common() if n >= least]
    return {token: k + 2 for k, token in enumerate(kept)}, counts

def encode(texts, vocab, split, length):
    """문장마다 토큰 번호를 length칸에 적는다. 남는 칸은 0, 넘치는 토큰은 버린다."""
    ids = torch.zeros(len(texts), length, dtype=torch.long)
    for row, text in enumerate(texts):
        found = [vocab.get(token, UNKNOWN) for token in split(text)][:length]
        ids[row, :len(found)] = torch.tensor(found, dtype=torch.long)
    return ids

sample = "배우들 연기는 좋았는데 너무 지루했다"
print(by_char(sample)[:8], len(by_char(sample)))
print(by_word(sample), len(by_word(sample)))
char_vocab, char_counts = make_vocab(texts_tr, by_char, least=5)
print(len(char_counts), len(char_vocab) + 2)
print(char_counts.most_common(4))
print(encode([sample], char_vocab, by_char, 24))

In [ ]:
def join(pieces, a, b):
    """이웃한 a, b를 한 조각으로 합친다."""
    out, k = [], 0
    while k < len(pieces):
        if k + 1 < len(pieces) and pieces[k] == a and pieces[k + 1] == b:
            out.append(a + b); k += 2
        else:
            out.append(pieces[k]); k += 1
    return out

def learn_merges(texts, merges):
    """가장 자주 이웃한 두 조각을 합치는 일을 merges번 한다. 어절 앞에는 _를 붙인다."""
    words = Counter("_" + word for text in texts for word in text.split())
    parts = {word: list(word) for word in words}     # 처음에는 글자 하나가 한 조각
    rules = []
    for _ in range(merges):
        pairs = Counter()
        for word, pieces in parts.items():
            for a, b in zip(pieces, pieces[1:]):
                pairs[a, b] += words[word]
        (a, b), n = pairs.most_common(1)[0]
        rules.append((a, b, n))
        for word, pieces in parts.items():
            if a in pieces and b in pieces:
                parts[word] = join(pieces, a, b)
    return rules

lap = time.perf_counter()
rules = learn_merges(texts_tr[:5000], 300)
print(f"{time.perf_counter() - lap:.0f}초")
print(rules[:4])
print(rules[-3:])
RANK = {(a, b): k for k, (a, b, n) in enumerate(rules)}
memo = {}

def split_word(word):
    """배운 순서대로 조각을 합친다. 같은 어절은 한 번만 계산한다."""
    if word not in memo:
        pieces = list("_" + word)
        while True:
            ranked = [(RANK[pair], k) for k, pair in enumerate(zip(pieces, pieces[1:]))
                      if pair in RANK]
            if not ranked:
                break
            k = min(ranked)[1]
            pieces[k:k + 2] = [pieces[k] + pieces[k + 1]]
        memo[word] = pieces
    return memo[word]

def by_piece(text):
    return [piece for word in text.split() for piece in split_word(word)]

print(by_piece(sample), len(by_piece(sample)))

In [ ]:
def pad(text, width):
    wide = sum(ord(ch) > 0x1100 for ch in text)
    return text + " " * max(width - len(text) - wide, 1)

sets, units = {}, {}
print("나누는 단위  서로 다른 토큰  어휘 크기  문장당 토큰  모르는 토큰  잘린 문장")
for name, split, least, length in [("글자", by_char, 5, 64), ("어절", by_word, 2, 24),
                                   ("부분 단어", by_piece, 2, 48)]:
    vocab, counts = make_vocab(texts_tr, split, least)
    X_tr = encode(texts_tr, vocab, split, length)
    X_va = encode(texts_va, vocab, split, length)
    each = np.array([len(split(text)) for text in texts_va])
    unknown = (X_va == UNKNOWN).sum().item() / (X_va != PAD).sum().item() * 100
    sets[name] = dict(vocab=vocab, counts=counts, split=split, length=length,
                      data=(X_tr, y_tr, X_va, y_va))
    units[name] = (len(counts), len(vocab) + 2, each.mean(), unknown,
                   (each > length).mean() * 100)
    print(pad(name, 12) + "{:>10,}  {:>10,}  {:>9.1f}  {:>10.2f}%  {:>8.1f}%".format(
        *units[name]))
for name in sets:
    ids = encode([sample], sets[name]["vocab"], sets[name]["split"], 10)
    print(pad(name, 10), ids[0].tolist())
for word in ["지루했다", "지루하다", "지루했음"]:
    print(word, sets["어절"]["vocab"].get(word, UNKNOWN), by_piece(word),
          encode([word], sets["부분 단어"]["vocab"], by_piece, 3)[0].tolist())

## 11.2 임베딩의 평균으로 문장을 벡터 하나로 만들기

In [ ]:
torch.manual_seed(0)
table = nn.Embedding(6, 2)                           # 토큰 6가지, 임베딩의 차원 2
ids = torch.tensor([[3, 5, 4, 2],                    # 토큰 4개인 문장
                    [3, 5, 0, 0]])                   # 토큰 2개인 문장. 뒤 두 칸은 패딩
vectors = table(ids)
print(vectors.shape)
print(table.weight.detach().numpy().round(2))
E = table.weight.detach().numpy()
print(E[[3, 5, 0, 0]].mean(axis=0).round(4), E[[3, 5]].mean(axis=0).round(4))

real = (ids != PAD)[:, :, None].float()              # 진짜 토큰 1, 패딩 0. (2, 4, 1)
masked = (vectors * real).sum(dim=1) / real.sum(dim=1)
print(vectors.mean(dim=1).detach().numpy().round(4))
print(masked.detach().numpy().round(4))

In [ ]:
class MeanNet(nn.Module):
    """토큰 임베딩의 평균을 선형층에 넣는다. mask가 참이면 패딩을 빼고 평균한다."""
    def __init__(self, vocab_size, mask=True, dim=32):
        super().__init__()
        self.table = nn.Embedding(vocab_size, dim)
        self.head = nn.Linear(dim, 2)
        self.mask = mask

    def pooled(self, ids):                           # ids: (배치, 길이)
        vectors = self.table(ids)                    # (배치, 길이, 32)
        if not self.mask:
            return vectors.mean(dim=1)
        real = (ids != PAD)[:, :, None].float()
        return (vectors * real).sum(dim=1) / real.sum(dim=1).clamp(min=1)

    def forward(self, ids):
        return self.head(self.pooled(ids))           # (배치, 2): 부정과 긍정의 점수

def accuracy(net, ids, y):
    net.eval()
    with torch.no_grad():
        guess = torch.cat([net(ids[k:k + 2000]).argmax(dim=1)
                           for k in range(0, len(ids), 2000)])
    return (guess == y).float().mean().item() * 100

def fit(net, ids, y, lr, seed=0, epochs=5):
    """Adam, 배치 크기 64, 교차 엔트로피 손실."""
    optimizer = torch.optim.Adam(net.parameters(), lr=lr)
    shuffler = torch.Generator().manual_seed(seed)
    for epoch in range(epochs):
        net.train()
        mixed = torch.randperm(len(ids), generator=shuffler)
        for start in range(0, len(ids), 64):
            rows = mixed[start:start + 64]
            optimizer.zero_grad()
            F.cross_entropy(net(ids[rows]), y[rows]).backward()
            optimizer.step()
    return net

count = lambda net: sum(p.numel() for p in net.parameters())
scores, nets, took = {}, {}, {}

def trial(name, make_net, data, lr, epochs=5, seeds=5):
    """시드를 바꿔 학습시키고 검증 정확도(%)를 scores[name]에 적는다."""
    X, y, X_test, y_test = data
    lap, scores[name], nets[name] = time.perf_counter(), [], []
    for seed in range(seeds):
        torch.manual_seed(seed)
        net = fit(make_net(), X, y, lr, seed, epochs)
        scores[name].append(accuracy(net, X_test, y_test))
        nets[name].append(net)
    scores[name], took[name] = np.array(scores[name]), time.perf_counter() - lap

def show(*names):
    print(pad("모델", 30) + "파라미터   정확도 (범위)           시간")
    for name in names:
        s = scores[name]
        span = f"({s.min():.2f}~{s.max():.2f})".ljust(15)
        print(pad(name, 30) + f"{count(nets[name][0]):>8,}   {np.median(s):>6.2f} {span}"
              f" {took[name]:>4.0f}초")

CHAR = sets["글자"]["data"]
V = len(char_vocab) + 2
print(V, CHAR[0].shape, CHAR[2].shape, count(MeanNet(V)))

## 11.3 기준선과 비교하고, 순서와 패딩을 확인하기

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

baseline = {}
print("글자 n-그램 로지스틱 회귀   특성 수    검증 정확도")
for longest in [1, 2, 3]:
    lap = time.perf_counter()
    grams = TfidfVectorizer(analyzer="char", ngram_range=(1, longest), min_df=2,
                            sublinear_tf=True)
    features = grams.fit_transform(texts_tr)         # 학습 자료로만 어휘를 정한다
    logistic = LogisticRegression(max_iter=1000, C=3).fit(features, y_tr.numpy())
    on_valid = logistic.score(grams.transform(texts_va), y_va.numpy()) * 100
    baseline[longest] = on_valid
    print(f"1~{longest}글자               {features.shape[1]:>9,}    {on_valid:.2f}   "
          f"{time.perf_counter() - lap:.1f}초")

In [ ]:
print("학습률     검증 정확도  (임베딩 평균, 글자, 시드 0)")
for lr in [0.001, 0.003, 0.01, 0.03]:
    torch.manual_seed(0)
    net = fit(MeanNet(V), CHAR[0], y_tr, lr)
    print(f"{lr:<8}   {accuracy(net, CHAR[2], y_va):.2f}")

In [ ]:
LR = 0.003
for name in ["글자", "어절", "부분 단어"]:
    size = len(sets[name]["vocab"]) + 2
    trial(f"임베딩 평균, {name}", lambda: MeanNet(size), sets[name]["data"], LR)
show("임베딩 평균, 글자", "임베딩 평균, 어절", "임베딩 평균, 부분 단어")

In [ ]:
first = nets["임베딩 평균, 글자"][0]
weight = first.head.weight.detach()
toward = weight[1] - weight[0]                       # 긍정 점수 - 부정 점수의 가중치
push = (first.table.weight.detach() @ toward).numpy()    # 글자마다 점수를 미는 정도
names_of = {k: token for token, k in char_vocab.items()}
often = [k for k in np.argsort(-push) if k > 1 and char_counts[names_of[k]] >= 150]
print("긍정 쪽:", " ".join(f"{names_of[k]}({push[k]:+.1f})" for k in often[:6]))
print("부정 쪽:", " ".join(f"{names_of[k]}({push[k]:+.1f})" for k in often[::-1][:6]))
for word in ["훌륭", "재밌", "짜증", "아깝"]:
    print(word, [f"{ch}({push[char_vocab[ch]]:+.1f})" for ch in word])

In [ ]:
class RecNet(nn.Module):
    """토큰 임베딩을 LSTM으로 차례로 읽고 마지막 토큰의 은닉 상태로 분류한다."""
    def __init__(self, vocab_size, dim=32, hidden=64):
        super().__init__()
        self.table = nn.Embedding(vocab_size, dim, padding_idx=PAD)
        self.recurrent = nn.LSTM(dim, hidden, batch_first=True)
        self.head = nn.Linear(hidden, 2)

    def forward(self, ids):
        states, _ = self.recurrent(self.table(ids))
        last = (ids != PAD).sum(dim=1).clamp(min=1) - 1     # 마지막 진짜 토큰의 위치
        return self.head(states[torch.arange(len(ids)), last])

trial("LSTM, 글자", lambda: RecNet(V), CHAR, 0.003, epochs=3)
show("임베딩 평균, 글자", "LSTM, 글자")

def positive(net, texts):
    """긍정일 확률."""
    net.eval()
    with torch.no_grad():
        scores_ = net(encode(texts, char_vocab, by_char, 64))
    return torch.softmax(scores_, dim=1)[:, 1].numpy()

pair = ["지루하지 않고 재미있다", "재미있지 않고 지루하다"]
print(sorted(pair[0]) == sorted(pair[1]))
print("임베딩 평균", positive(first, pair).round(4))
print("LSTM       ", positive(nets["LSTM, 글자"][0], pair).round(4))

In [ ]:
rng = np.random.default_rng(0)
cut = [text[:64] for text in texts_va]                    # 앞 64글자만 쓴다
mixed_up = ["".join(rng.permutation(list(text))) for text in cut]   # 글자 순서를 섞는다
print(sample, "->", "".join(rng.permutation(list(sample))))
X_cut, X_mixed = (encode(texts, char_vocab, by_char, 64) for texts in (cut, mixed_up))
shuffled = {}
print("검증 정확도(%)             원래 문장   글자를 섞은 문장")
for name in ["임베딩 평균, 글자", "LSTM, 글자"]:
    before = np.median([accuracy(net, X_cut, y_va) for net in nets[name]])
    after = np.median([accuracy(net, X_mixed, y_va) for net in nets[name]])
    shuffled[name] = (before, after)
    print(pad(name, 26) + f"{before:.2f}       {after:.2f}")
before = logistic.score(grams.transform(cut), y_va.numpy()) * 100
after = logistic.score(grams.transform(mixed_up), y_va.numpy()) * 100
shuffled["로지스틱 회귀, 1~3글자"] = (before, after)
print(pad("로지스틱 회귀, 1~3글자", 26) + f"{before:.2f}       {after:.2f}")

In [ ]:
trial("임베딩 평균, 패딩 포함", lambda: MeanNet(V, mask=False), CHAR, LR)
show("임베딩 평균, 글자", "임베딩 평균, 패딩 포함")

def widen(ids, length):
    """오른쪽에 패딩을 더 붙여 length칸으로 만든다."""
    extra = torch.zeros(len(ids), length - ids.shape[1], dtype=torch.long)
    return torch.cat([ids, extra], dim=1)

by_length = {}
print("패딩까지의 길이          64      96     128     192     256")
for name in ["임베딩 평균, 글자", "임베딩 평균, 패딩 포함"]:
    by_length[name] = np.array([[accuracy(net, widen(CHAR[2], length), y_va)
                                 for length in [64, 96, 128, 192, 256]]
                                for net in nets[name]])
    middle = np.median(by_length[name], axis=0)
    print(pad(name, 22) + "".join(f"{v:>8.2f}" for v in middle))

one = encode(["재미있어요"], char_vocab, by_char, 8)
loose = nets["임베딩 평균, 패딩 포함"][0]
for length in [8, 64, 256]:
    with torch.no_grad():
        a = torch.softmax(first(widen(one, length)), dim=1)[0, 1].item()
        b = torch.softmax(loose(widen(one, length)), dim=1)[0, 1].item()
    print(f"길이 {length:>3}: 패딩을 뺀 평균 {a:.4f}   패딩을 넣은 평균 {b:.4f}")

In [ ]:
both = texts_tr + texts_va
wide_vocab, _ = make_vocab(both, by_word, 2)              # 검증 문장까지 넣어 만든 어휘
wide = (encode(texts_tr, wide_vocab, by_word, 24), y_tr,
        encode(texts_va, wide_vocab, by_word, 24), y_va)
unknown = (wide[2] == UNKNOWN).sum().item() / (wide[2] != PAD).sum().item() * 100
print(len(sets["어절"]["vocab"]) + 2, len(wide_vocab) + 2, round(unknown, 2))
trial("어절, 검증 문장까지 넣은 어휘", lambda: MeanNet(len(wide_vocab) + 2), wide, LR)
show("임베딩 평균, 어절", "어절, 검증 문장까지 넣은 어휘")
print(scores["임베딩 평균, 어절"].round(2))
print(scores["어절, 검증 문장까지 넣은 어휘"].round(2))

## 11.4 어텐션을 넘파이로 계산하기: 질의, 키, 값

In [ ]:
def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)
    return np.exp(z) / np.exp(z).sum(axis=-1, keepdims=True)

def attention(Q, K, V_, hide=None):
    """Q: (질의 수, d), K: (키 수, d), V_: (키 수, 값의 길이). hide: 가릴 키는 True."""
    scores = Q @ K.T / np.sqrt(K.shape[1])               # (질의 수, 키 수)
    if hide is not None:
        scores = np.where(hide, -np.inf, scores)         # 가린 키의 점수를 -무한대로
    weights = softmax(scores)                            # 행마다 합이 1
    return weights @ V_, weights

Q = np.array([[2.0, 2.0, 0.0, 0.0]])                     # 질의 1개
K = np.array([[1.0, 1.0, 0.0, 0.0],                      # 키 3개
              [1.0, 0.0, 1.0, 0.0],
              [0.0, 0.0, 1.0, 1.0]])
values = np.array([[1.0, 0.0], [0.0, 1.0], [5.0, 5.0]])  # 키마다 값 하나
out, weights = attention(Q, K, values)
print((Q @ K.T), weights.round(4), out.round(4))
by_torch = F.scaled_dot_product_attention(*map(torch.tensor, (Q, K, values)))
print(float(np.abs(by_torch.numpy() - out).max()))
out, weights = attention(Q, K, values, hide=np.array([False, False, True]))
print(weights.round(4), out.round(4))

## 11.5 답을 아는 과제에서 어텐션이 보는 곳 확인하기

In [ ]:
def make_task(n, seed):
    """색 8가지 중 6가지에 숫자(0~9)를 하나씩 준다. 답은 질문한 색의 숫자다."""
    rng = np.random.default_rng(seed)
    colors = np.array([rng.permutation(8)[:6] for _ in range(n)])   # (n, 6) 다른 색
    numbers = rng.integers(0, 10, (n, 6))
    where = rng.integers(0, 6, n)                                   # 답이 있는 칸
    asked = colors[np.arange(n), where]
    ids = np.column_stack([80 + asked, colors * 10 + numbers])      # 질문 1 + 항목 6
    answer = numbers[np.arange(n), where]
    return torch.tensor(ids), torch.tensor(answer), torch.tensor(where), numbers

T_tr, a_tr, _, _ = make_task(20000, seed=0)
T_te, a_te, where_te, numbers_te = make_task(5000, seed=1)
TASK = (T_tr, a_tr, T_te, a_te)
print(T_tr.shape, T_tr[0].tolist(), a_tr[0].item())
print([f"색{k // 10} 숫자{k % 10}" for k in T_tr[0, 1:].tolist()])
print("질문: 색", T_tr[0, 0].item() - 80)

In [ ]:
class TaskMean(nn.Module):
    """7칸의 임베딩을 평균해서 분류한다. hidden을 주면 은닉층을 하나 둔다."""
    def __init__(self, hidden=0, dim=32):
        super().__init__()
        self.table = nn.Embedding(88, dim)
        self.head = nn.Linear(dim, 10)
        if hidden:
            self.head = nn.Sequential(nn.Linear(dim, hidden), nn.ReLU(),
                                      nn.Linear(hidden, 10))

    def forward(self, ids):
        return self.head(self.table(ids).mean(dim=1))

class TaskAttention(nn.Module):
    """질문 칸을 질의로, 항목 6칸을 키와 값으로 쓰는 어텐션 한 번."""
    def __init__(self, dim=32):
        super().__init__()
        self.table = nn.Embedding(88, dim)
        self.to_query = nn.Linear(dim, dim, bias=False)
        self.to_key = nn.Linear(dim, dim, bias=False)
        self.to_value = nn.Linear(dim, dim, bias=False)
        self.head = nn.Linear(dim, 10)

    def weights(self, ids):
        vectors = self.table(ids)                            # (배치, 7, 32)
        query = self.to_query(vectors[:, :1])                # (배치, 1, 32)
        keys = self.to_key(vectors[:, 1:])                   # (배치, 6, 32)
        scores = query @ keys.transpose(1, 2) / keys.shape[-1] ** 0.5
        return torch.softmax(scores, dim=-1)[:, 0], vectors  # (배치, 6)

    def forward(self, ids):
        w, vectors = self.weights(ids)
        return self.head((w[:, :, None] * self.to_value(vectors[:, 1:])).sum(dim=1))

trial("평균 + 선형층", lambda: TaskMean(), TASK, 0.003, epochs=10)
trial("평균 + 은닉층 64", lambda: TaskMean(hidden=64), TASK, 0.003, epochs=10)
trial("어텐션", lambda: TaskAttention(), TASK, 0.003, epochs=10)
show("평균 + 선형층", "평균 + 은닉층 64", "어텐션")

most = np.array([np.bincount(row, minlength=10).argmax() for row in numbers_te])
print(round((most == a_te.numpy()).mean() * 100, 2))     # 가장 많이 나온 숫자로 답할 때
on_answer, right = [], []
for net in nets["어텐션"]:
    with torch.no_grad():
        w, _ = net.weights(T_te)
    on_answer.append(w[torch.arange(len(T_te)), where_te].mean().item())
    right.append((w.argmax(dim=1) == where_te).float().mean().item() * 100)
print(np.round(on_answer, 3), np.round(right, 1))

In [ ]:
class AttentionPool(nn.Module):
    """학습되는 질의 하나로 토큰마다 가중치를 구해 임베딩을 가중 평균한다."""
    def __init__(self, vocab_size, dim=32):
        super().__init__()
        self.table = nn.Embedding(vocab_size, dim)
        self.query = nn.Parameter(torch.zeros(dim))
        self.head = nn.Linear(dim, 2)

    def weights(self, ids):
        vectors = self.table(ids)                            # 키와 값으로 함께 쓴다
        scores = vectors @ self.query / vectors.shape[-1] ** 0.5
        scores = scores.masked_fill(ids == PAD, float("-inf"))   # 패딩은 가린다
        return torch.softmax(scores, dim=1), vectors

    def forward(self, ids):
        w, vectors = self.weights(ids)
        return self.head((w[:, :, None] * vectors).sum(dim=1))

trial("어텐션으로 모으기, 글자", lambda: AttentionPool(V), CHAR, LR)
show("임베딩 평균, 글자", "어텐션으로 모으기, 글자", "LSTM, 글자")
pooler = nets["어텐션으로 모으기, 글자"][0]
looked = {}
examples = ["배우들 연기는 좋았는데 내용이 너무 지루했다",
            "기대 안 했는데 정말 재밌게 봤어요"]
for text in examples:
    ids = encode([text], char_vocab, by_char, 64)
    with torch.no_grad():
        w = pooler.weights(ids)[0][0, :len(text)].numpy()
    looked[text] = w
    top = np.argsort(-w)[:5]
    print(text)
    chance = positive(pooler, [text])[0]
    print(f"  긍정 확률 {chance:.2f}, 고르게 나눈 가중치 {1 / len(text):.3f}")
    print("  " + "  ".join(f"{text[k]}({w[k]:.3f})" for k in top))
print("어텐션으로 모으기", positive(pooler, pair).round(4))

In [ ]:
import sys, platform, sklearn

print(sys.version.split()[0], np.__version__, pd.__version__, torch.__version__,
      sklearn.__version__)
print(platform.machine(), torch.get_default_dtype(), torch.get_num_threads())

In [ ]:
!pip -q install koreanize-matplotlib

In [ ]:
import matplotlib.pyplot as plt
import koreanize_matplotlib
from matplotlib.colors import to_rgba
from matplotlib.ticker import FuncFormatter

NAVY, BRICK, TEAL, GRAY = "#16324f", "#b5532a", "#2a7f73", "#9aa1ab"
BOX = {"fc": "white", "ec": "none", "alpha": 0.9, "pad": 1.5}


def frame(ax, title=None, color=NAVY, grid="y"):
    """그래프마다 반복하는 꾸밈: 제목, 보조선, 테두리 정리."""
    if title:
        ax.set_title(title, fontsize=9, loc="left", color=color, weight="bold")
    ax.grid(axis=grid, color="#e3e7ec", lw=0.6)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=8)
    ax.spines[["top", "right"]].set_visible(False)


def plain_ticks(axis):
    """로그 축의 눈금을 10의 거듭제곱 대신 보통 숫자로 적는다."""
    axis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}" if val >= 1 else f"{val:.7f}".rstrip("0")))

In [ ]:
# 그림 11-2
fig, axes = plt.subplots(3, 1, figsize=(7, 8.6))
unit_names = ["글자", "어절", "부분 단어"]
unit_colors = [NAVY, BRICK, TEAL]
panels = [(1, "① 어휘 크기: 모델이 아는 토큰의 가짓수 (패딩과 '모르는 토큰' 포함)", "{:,.0f}개"),
          (2, "② 검증 문장 하나의 토큰 수 (평균)", "{:.1f}개"),
          (3, "③ 검증 문장의 토큰 가운데 어휘에 없는 토큰의 비율", "{:.2f}%")]
remarks = [{1: f"글자의 {units['어절'][1] / units['글자'][1]:.0f}배. 임베딩 표의 행이 그만큼 많다"},
           {0: f"어절의 {units['글자'][2] / units['어절'][2]:.1f}배. 문장이 그만큼 길어진다"},
           {1: "토큰 열 개 가운데 네 개를 모델이 알아보지 못한다"}]
for ax, (col, title, form), remark in zip(axes, panels, remarks):
    amounts = [units[name][col] for name in unit_names]
    ax.barh(unit_names, amounts, color=unit_colors, height=0.58)
    for row, amount in enumerate(amounts):
        ax.text(amount + max(amounts) * 0.012, row, form.format(amount), va="center", fontsize=10,
                color=unit_colors[row], weight="bold")
        if row in remark:
            inside = amount > max(amounts) * 0.6
            ax.text(amount * 0.5 if inside else amount + max(amounts) * 0.17, row, remark[row], va="center",
                    ha="center" if inside else "left", fontsize=9, color="white" if inside else "#333")
    ax.set_xlim(0, max(amounts) * 1.17)
    ax.invert_yaxis()
    frame(ax, grid="x")
    ax.set_title(title, fontsize=10, loc="left", color=NAVY, weight="bold")
    ax.tick_params(axis="y", labelsize=10.5, length=0)
    ax.tick_params(axis="x", labelsize=9)
    ax.xaxis.set_major_formatter(FuncFormatter(lambda val, _: f"{val:,.0f}"))
fig.subplots_adjust(left=0.14, right=0.97, top=0.96, bottom=0.04, hspace=0.42)
fig.savefig("fig_11_2.png", dpi=300)
plt.show()
print({name: [round(float(v), 2) for v in units[name]] for name in unit_names})

In [ ]:
# 그림 11-4
fig, axes = plt.subplots(2, 1, figsize=(7, 9.0), gridspec_kw={"height_ratios": [2.5, 1]})
ax = axes[0]
plus, minus = often[:12], often[::-1][:12]
shown = list(plus) + list(minus[::-1])
spots = np.arange(len(shown))
bar_colors = [TEAL] * 12 + [BRICK] * 12
ax.barh(spots, push[shown], color=bar_colors, height=0.74)
for spot, k in zip(spots, shown):
    side = 1 if push[k] > 0 else -1
    ax.text(push[k] + side * 0.8, spot, f"{push[k]:+.1f}".replace("-", "$-$"), va="center",
            ha="left" if side > 0 else "right", fontsize=9, color=bar_colors[spot])
    ax.text(-side * 1.4, spot, names_of[k], va="center", ha="right" if side > 0 else "left", fontsize=11.5,
            color="#111", weight="bold")
    ax.text(-side * 6.2, spot, f"{char_counts[names_of[k]]:,}번", va="center", ha="right" if side > 0 else "left",
            fontsize=9, color="#6b7480")
ax.axvline(0, color="#444", lw=0.9)
ax.axhline(11.5, color="#c9ced6", lw=0.8, ls="--")
ax.set_yticks([]); ax.invert_yaxis(); ax.set_xlim(-50, 50)
ax.set_xticks(range(-40, 41, 20)); ax.set_xticklabels([f"{v:+d}".replace("-", "$-$") if v else "0" for v in range(-40, 41, 20)])
ax.set_xlabel("글자 하나가 '긍정 점수 $-$ 부정 점수'에 더하는 값", fontsize=9)
ax.text(-48, 5.5, "긍정 쪽으로\n가장 크게 미는\n글자 12개", ha="left", va="center", fontsize=10, color=TEAL, weight="bold",
        linespacing=1.5)
ax.text(48, 17.5, "부정 쪽으로\n가장 크게 미는\n글자 12개", ha="right", va="center", fontsize=10, color=BRICK, weight="bold",
        linespacing=1.5)
ax.text(49, 23.9, "긍정 →", fontsize=9, color=TEAL, ha="right", va="bottom")
ax.text(-49, 23.9, "← 부정", fontsize=9, color=BRICK, ha="left", va="bottom")
frame(ax, grid="x")
ax.set_title("① 글자별 점수 (회색 숫자는 학습 문장에 나온 횟수. 150번 이상 나온 글자만)", fontsize=10, loc="left",
             color=NAVY, weight="bold")
ax.tick_params(axis="x", labelsize=9)
ax = axes[1]
four_words = ["훌륭", "재밌", "짜증", "아깝"]
place = 0
for word in four_words:
    for ch in word:
        value = push[char_vocab[ch]]
        ax.bar(place, value, width=0.7, color=TEAL if value > 0 else BRICK)
        ax.text(place, value + (2.2 if value > 0 else -2.2), f"{value:+.1f}".replace("-", "$-$"), ha="center",
                va="bottom" if value > 0 else "top", fontsize=9, color=TEAL if value > 0 else BRICK)
        ax.text(place, -58, ch, ha="center", va="center", fontsize=13, weight="bold", color="#111")
        place += 1
    ax.text(place - 1.5, 52, f"‘{word}’", ha="center", fontsize=10, color=NAVY, weight="bold")
    place += 0.8
ax.axhline(0, color="#444", lw=0.9)
ax.set_xticks([]); ax.set_ylim(-66, 62); ax.set_xlim(-0.8, place - 0.9)
ax.set_yticks([-40, -20, 0, 20, 40]); ax.set_yticklabels(["$-$40", "$-$20", "0", "+20", "+40"])
frame(ax)
ax.set_title("② 낱말 네 개를 글자로 나눴을 때의 점수: 여러 낱말에 쓰이는 글자(재, 아)는 0에 가깝다", fontsize=10,
             loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
fig.subplots_adjust(left=0.07, right=0.97, top=0.965, bottom=0.03, hspace=0.24)
fig.savefig("fig_11_4.png", dpi=300)
plt.show()
print([names_of[k] for k in plus], [names_of[k] for k in minus])

In [ ]:
# 그림 11-5
from matplotlib.patches import Patch, Rectangle
fig, axes = plt.subplots(3, 1, figsize=(7, 9.4), gridspec_kw={"height_ratios": [0.62, 1.25, 1]})
ax = axes[0]
tile_rng = np.random.default_rng(1)
tile_text = "배우들 연기는 좋았는데 너무 지루했다"
tile_order = tile_rng.permutation(len(tile_text))
for row, (label, letters, shade) in enumerate([("원래 문장", list(tile_text), NAVY),
                                              ("글자를 섞은 문장", [tile_text[k] for k in tile_order], BRICK)]):
    for k, ch in enumerate(letters):
        ax.add_patch(Rectangle((k, 1.3 - row * 1.3), 0.92, 0.92, fc="#eef1f5" if row == 0 else "#f7ece6", ec=shade, lw=0.9))
        ax.text(k + 0.46, 1.76 - row * 1.3, ch, ha="center", va="center", fontsize=10.5, color=shade, weight="bold")
    ax.text(-0.4, 1.76 - row * 1.3, label, ha="right", va="center", fontsize=9, color=shade, weight="bold")
ax.set_xlim(-5.2, len(tile_text) + 0.2); ax.set_ylim(-0.25, 2.45); ax.axis("off")
ax.set_title("① 섞는 방법: 문장 안에서 글자의 차례만 바꾼다. 들어 있는 글자는 같다", fontsize=10, loc="left", color=NAVY,
             weight="bold")
ax = axes[1]
trio = ["임베딩 평균, 글자", "LSTM, 글자", "로지스틱 회귀, 1~3글자"]
for spot, label in enumerate(trio):
    kept, mixed = shuffled[label]
    ax.bar(spot - 0.19, kept, width=0.36, color=NAVY)
    ax.bar(spot + 0.19, mixed, width=0.36, color=BRICK)
    ax.text(spot - 0.19, kept + 0.25, f"{kept:.2f}", ha="center", fontsize=9.6, color=NAVY, weight="bold")
    ax.text(spot + 0.19, mixed + 0.25, f"{mixed:.2f}", ha="center", fontsize=9.6, color=BRICK, weight="bold")
    gap = mixed - kept
    if abs(gap) < 0.005:
        ax.text(spot, 71.2, "변화 없음", ha="center", fontsize=10, color="#111", weight="bold", bbox=BOX)
    else:
        ax.annotate("", xy=(spot + 0.19, mixed + 1.3), xytext=(spot + 0.19, kept - 0.2),
                    arrowprops=dict(arrowstyle="-|>", color="#444", lw=1.1))
        ax.text(spot + 0.25, (kept + mixed) / 2 + 0.6, f"{gap:+.2f}\n%p".replace("-", "$-$"), ha="left", va="center", fontsize=9.6, linespacing=1.2,
                color="#111", weight="bold")
ax.set_xticks(range(3)); ax.set_xticklabels(trio, fontsize=10); ax.set_xlim(-0.5, 2.95)
ax.set_ylim(70, 88.5); ax.set_ylabel("검증 정확도 (%)", fontsize=9)
ax.legend(handles=[Patch(color=NAVY, label="원래 문장"), Patch(color=BRICK, label="글자를 섞은 문장")], frameon=False,
          fontsize=9.4, loc="upper left", ncol=2)
frame(ax)
ax.set_title("② 검증 문장 10,000개의 정확도 (세로축은 70%에서 시작)", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
ax = axes[2]
both_ways = {"임베딩 평균": positive(first, pair), "LSTM": positive(nets["LSTM, 글자"][0], pair)}
for spot, (label, chances) in enumerate(both_ways.items()):
    for shift, chance, shade in [(-0.19, chances[0], TEAL), (0.19, chances[1], BRICK)]:
        ax.bar(spot + shift, chance, width=0.36, color=shade)
        ax.text(spot + shift, chance + 0.025, f"{chance:.3f}", ha="center", fontsize=9.6, color=shade, weight="bold")
        verdict = "긍정으로 분류" if chance > 0.5 else "부정으로 분류"
        ax.text(spot + shift, -0.085, verdict, ha="center", fontsize=9, color=shade)
ax.axhline(0.5, color="#666", lw=0.9, ls="--")
ax.text(1.49, 0.515, "0.5", fontsize=9, color="#555", ha="right")
ax.text(0, 0.62, "두 문장을 구별하지 못한다\n(한 문장은 반드시 틀린다)", ha="center", fontsize=9, color="#111", linespacing=1.5)
ax.set_xticks(range(2)); ax.set_xticklabels(list(both_ways), fontsize=10); ax.set_xlim(-0.5, 1.5)
ax.tick_params(axis="x", pad=24, length=0)
ax.set_ylim(0, 1.16); ax.set_ylabel("긍정일 확률", fontsize=9)
ax.legend(handles=[Patch(color=TEAL, label=f"“{pair[0]}” (긍정인 문장)"), Patch(color=BRICK, label=f"“{pair[1]}” (부정인 문장)")],
          frameon=False, fontsize=9, loc="upper left", ncol=1, bbox_to_anchor=(0.0, 1.0))
frame(ax)
ax.set_title("③ 같은 글자로 만든, 뜻이 반대인 두 문장에 모델이 준 확률", fontsize=10, loc="left", color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
fig.subplots_adjust(left=0.1, right=0.97, top=0.965, bottom=0.07, hspace=0.3)
fig.savefig("fig_11_5.png", dpi=300)
plt.show()
print({label: [round(float(v), 4) for v in chances] for label, chances in both_ways.items()})

In [ ]:
# 그림 11-6
fig, axes = plt.subplots(2, 1, figsize=(7, 9.0), gridspec_kw={"height_ratios": [1.25, 1]})
ax = axes[0]
lengths = [64, 96, 128, 192, 256]
for label, shade, mark, word in [("임베딩 평균, 글자", NAVY, "o", "패딩을 빼고 평균한 모델 (마스크 사용)"),
                                 ("임베딩 평균, 패딩 포함", BRICK, "s", "패딩까지 넣어 평균한 모델")]:
    table_ = by_length[label]
    middle = np.median(table_, axis=0)
    ax.fill_between(lengths, table_.min(axis=0), table_.max(axis=0), color=shade, alpha=0.14, lw=0)
    ax.plot(lengths, middle, color=shade, marker=mark, ms=7, lw=2.2, label=word)
    for length, value in zip(lengths, middle):
        ax.text(length, value + (0.75 if shade == NAVY else -1.15), f"{value:.2f}", ha="center", fontsize=9.6,
                color=shade, weight="bold")
worst = by_length["임베딩 평균, 패딩 포함"][:, -1].min()
ax.annotate(f"가장 나쁜 시드 {worst:.2f}", xy=(256, worst), xytext=(214, worst + 0.2), fontsize=9, color=BRICK,
            va="center", ha="right", arrowprops=dict(arrowstyle="->", color=BRICK, lw=0.9))
ax.axvline(64, color="#888", lw=0.9, ls=":")
ax.text(67, 81.6, "학습할 때의 길이 64칸.\n여기서는 두 모델이 구별되지 않는다", fontsize=9, color="#444", va="center", linespacing=1.5)
ax.set_xticks(lengths); ax.set_xticklabels([f"{length}칸\n(패딩 +{length - 64})" if length > 64 else "64칸\n(학습할 때)" for length in lengths],
                                           fontsize=9)
ax.set_xlim(52, 268); ax.set_ylim(62, 83.5)
ax.set_ylabel("검증 정확도 (%)", fontsize=9)
ax.legend(frameon=False, fontsize=9.4, loc="lower left")
frame(ax)
ax.set_title("① 검증 문장 10,000개에 패딩만 더 붙였을 때 (선은 시드 5개의 중앙값, 띠는 최소~최대)", fontsize=10, loc="left",
             color=NAVY, weight="bold")
ax.tick_params(axis="y", labelsize=9)
ax = axes[1]
one_text = encode(["재미있어요"], char_vocab, by_char, 8)
steps = [8, 16, 32, 64, 128, 256]
curves = {}
for label, net, shade, mark in [("패딩을 빼고 평균한 모델", first, NAVY, "o"),
                                ("패딩까지 넣어 평균한 모델", nets["임베딩 평균, 패딩 포함"][0], BRICK, "s")]:
    with torch.no_grad():
        curves[label] = [torch.softmax(net(widen(one_text, length)), dim=1)[0, 1].item() for length in steps]
    ax.plot(range(len(steps)), curves[label], color=shade, marker=mark, ms=7, lw=2.2, label=label)
    for spot, value in enumerate(curves[label]):
        ax.text(spot, value + (0.05 if shade == NAVY else -0.085), f"{value:.3f}", ha="center", fontsize=9.4, color=shade,
                weight="bold")
ax.axhline(0.5, color="#666", lw=0.9, ls="--")
ax.axhspan(0.5, 1.2, color=TEAL, alpha=0.05, lw=0); ax.axhspan(0, 0.5, color=BRICK, alpha=0.05, lw=0)
ax.text(-0.4, 0.53, "↑ 긍정으로 분류", fontsize=9, color=TEAL, ha="left"); ax.text(-0.4, 0.44, "↓ 부정으로 분류", fontsize=9, color=BRICK, ha="left")
ax.set_xticks(range(len(steps))); ax.set_xticklabels([f"{length}칸\n(글자 5 + 패딩 {length - 5})" for length in steps], fontsize=9)
ax.set_xlim(-0.45, 5.45); ax.set_ylim(0.22, 1.14); ax.set_ylabel("긍정일 확률", fontsize=9)
frame(ax)
ax.set_title("② “재미있어요” 한 문장을 길이가 다른 칸에 넣었을 때 (시드 0의 모델)", fontsize=10, loc="left", color=NAVY,
             weight="bold")
ax.tick_params(axis="y", labelsize=9)
fig.subplots_adjust(left=0.1, right=0.97, top=0.965, bottom=0.065, hspace=0.3)
fig.savefig("fig_11_6.png", dpi=300)
plt.show()
print({label: np.median(by_length[label], axis=0).round(2).tolist() for label in by_length})
print(by_length["임베딩 평균, 패딩 포함"][:, -1].round(2))
print({label: [round(v, 4) for v in values_] for label, values_ in curves.items()})

In [ ]:
# 그림 11-9
from matplotlib.patches import Rectangle
SWATCH = ["#16324f", "#b5532a", "#2a7f73", "#d9a441", "#7b5ea7", "#5b9bd5", "#c2567a", "#6b7f3a"]   # 색0~색7
fig, axes = plt.subplots(2, 1, figsize=(7, 9.4), gridspec_kw={"height_ratios": [2.9, 1]})
ax = axes[0]
picked = np.arange(10)                                              # 시험 문제의 앞 10개
with torch.no_grad():
    seen, _ = nets["어텐션"][0].weights(T_te[picked])
    replies = nets["어텐션"][0](T_te[picked]).argmax(dim=1).numpy()
seen = seen.numpy()
ax.imshow(seen, cmap="Blues", vmin=0, vmax=1.25, aspect="auto", extent=(-0.5, 5.5, 9.5, -0.5))
for row in range(10):
    asked_color = T_te[picked[row], 0].item() - 80
    ax.add_patch(Rectangle((-1.36, row - 0.2), 0.3, 0.4, fc=SWATCH[asked_color], ec="none", clip_on=False))
    ax.text(-1.0, row, f"색{asked_color}", va="center", fontsize=9.6, color="#111", clip_on=False)
    for col in range(6):
        item = T_te[picked[row], col + 1].item()
        is_answer = col == where_te[picked[row]].item()
        ink = "white" if seen[row, col] > 0.6 else "#222"
        ax.add_patch(Rectangle((col - 0.36, row - 0.34), 0.2, 0.26, fc=SWATCH[item // 10], ec="white", lw=0.6))
        ax.text(col - 0.1, row - 0.2, f"색{item // 10}: {item % 10}", ha="left", va="center", fontsize=9, color=ink)
        ax.text(col, row + 0.2, f"{seen[row, col]:.2f}", ha="center", va="center", fontsize=10 if is_answer else 8.6,
                weight="bold" if is_answer else "normal", color=ink)
        if is_answer:
            ax.add_patch(Rectangle((col - 0.48, row - 0.46), 0.96, 0.92, fill=False, ec=BRICK, lw=2.2))
    right_answer = a_te[picked[row]].item()
    ax.text(5.72, row, f"{replies[row]}", va="center", fontsize=11, color=TEAL if replies[row] == right_answer else BRICK,
            weight="bold", clip_on=False)
    ax.text(6.12, row, f"(정답 {right_answer})", va="center", fontsize=9, color="#555", clip_on=False)
ax.set_xticks(range(6)); ax.set_xticklabels([f"항목 {k}" for k in range(1, 7)], fontsize=9.6)
ax.xaxis.tick_top(); ax.set_yticks([]); ax.tick_params(length=0)
ax.text(-1.36, -0.78, "질문", fontsize=9.6, color=NAVY, weight="bold", clip_on=False)
ax.text(-0.5, -1.14, "칸의 윗줄은 항목(색: 숫자), 아랫줄은 그 항목이 받은 가중치다. 가중치가 클수록 칸이 진하다", fontsize=9, color="#444", clip_on=False)
ax.text(5.72, -0.78, "모델의 답", fontsize=9.6, color=NAVY, weight="bold", clip_on=False)
for side in ax.spines.values():
    side.set_visible(False)
ax.set_xlim(-0.5, 5.5); ax.set_ylim(9.5, -0.5)
ax.set_title("① 시험 문제 10개의 어텐션 가중치. 굵은 숫자와 벽돌색 테두리가 답이 있는 칸이다", fontsize=10, loc="left",
             color=NAVY, weight="bold", pad=40, x=-0.2)
ax = axes[1]
rule = (np.array([np.bincount(row, minlength=10).argmax() for row in numbers_te]) == a_te.numpy()).mean() * 100
trio = [("평균 + 선형층", GRAY), ("평균 + 은닉층 64", TEAL), ("어텐션", NAVY)]
for row, (label, shade) in enumerate(trio):
    marks = scores[label]
    ax.barh(row, np.median(marks), color=shade, height=0.58)
    ax.scatter(marks, [row] * len(marks), s=18, color="white", edgecolor="#222", lw=0.7, zorder=5)
    ax.text(max(marks) + 1.8, row, f"{np.median(marks):.2f}%", va="center", fontsize=10.5, weight="bold")
ax.axvline(10, color="#666", lw=0.9, ls="--"); ax.axvline(rule, color=BRICK, lw=0.9, ls="--")
ax.text(10.8, -0.56, "아무 숫자나\n답할 때 10%", fontsize=9, color="#555", va="center", linespacing=1.3)
ax.text(rule + 1.2, -0.56, f"항목에 가장 많이 나온 숫자를\n답하는 규칙 {rule:.2f}%", fontsize=9, color=BRICK, va="center", linespacing=1.3)
ax.set_yticks(range(3)); ax.set_yticklabels([label for label, _ in trio], fontsize=10)
ax.set_ylim(2.5, -0.95); ax.set_xlim(0, 116); ax.set_xlabel("시험 문제 5,000개의 정확도 (%)", fontsize=9)
frame(ax, grid="x")
ax.set_title("② 세 모델의 정확도 (막대는 시드 5개의 중앙값, 흰 점은 시드 하나의 값)", fontsize=10, loc="left", color=NAVY,
             weight="bold", x=-0.2)
ax.tick_params(axis="x", labelsize=9); ax.tick_params(axis="y", length=0)
fig.subplots_adjust(left=0.19, right=0.86, top=0.9, bottom=0.055, hspace=0.16)
fig.savefig("fig_11_9.png", dpi=300)
plt.show()
print(seen[np.arange(10), where_te[picked].numpy()].round(2), seen.sum(axis=1).round(3))
print(replies.tolist(), a_te[picked].tolist(), round(float(rule), 2))

In [ ]:
# 그림 11-10
fig, axes = plt.subplots(2, 1, figsize=(7, 8.2))
remarks = [("좋", "‘좋’과 ‘루’의 가중치가 거의 같은데\n모델의 답은 부정이다"), ("안", "‘안’이 두 번째로 크다.\n긍정의 근거인지는 알 수 없다")]
for ax, text, (mark_ch, remark), order_no in zip(axes, examples, remarks, "①②"):
    given = looked[text]
    spots = np.arange(len(text))
    level = 1 / len(text)
    ax.bar(spots, given, color=[NAVY if v > level else "#c3cad3" for v in given], width=0.82)
    ax.axhline(level, color=BRICK, lw=1.1, ls="--")
    ax.text(-0.5, max(given) * 1.5, f"점선: 고르게 나눈 가중치 1/{len(text)} = {level:.3f}", fontsize=9, color=BRICK, ha="left")
    for spot in np.argsort(-given)[:3]:
        ax.text(spot, given[spot] + max(given) * 0.02, f"{given[spot]:.3f}", ha="center", fontsize=9, color=NAVY, weight="bold")
    where_mark = text.index(mark_ch)
    ax.annotate(remark, xy=(where_mark, given[where_mark] + max(given) * 0.09), xytext=(where_mark + 2.6, max(given) * 1.27),
                fontsize=9, color="#111", ha="left", va="center", linespacing=1.45,
                arrowprops=dict(arrowstyle="->", color="#444", lw=0.9))
    ax.set_xticks(spots); ax.set_xticklabels(["" if ch == " " else ch for ch in text], fontsize=12.5, weight="bold")
    for spot, ch in enumerate(text):
        if ch == " ":
            ax.text(spot, -max(given) * 0.085, "빈칸", ha="center", va="center", fontsize=6.6, color="#8a929c")
    ax.tick_params(axis="x", length=0); ax.tick_params(axis="y", labelsize=9)
    ax.set_xlim(-0.7, len(text) - 0.3); ax.set_ylim(0, max(given) * 1.6)
    ax.set_ylabel("어텐션 가중치", fontsize=9)
    chance = positive(pooler, [text])[0]
    frame(ax)
    ax.set_title(f"{order_no} “{text}”", fontsize=10.5, loc="left", color=NAVY, weight="bold")
    ax.text(1.0, 1.035, f"모델의 답: {'긍정' if chance > 0.5 else '부정'} (긍정 확률 {chance:.2f})", transform=ax.transAxes, ha="right",
            fontsize=9.4, color=TEAL if chance > 0.5 else BRICK, weight="bold")
fig.subplots_adjust(left=0.1, right=0.97, top=0.955, bottom=0.05, hspace=0.32)
fig.savefig("fig_11_10.png", dpi=300)
plt.show()
print({text: (round(float(looked[text].max()), 3), text[int(looked[text].argmax())],
              round(float(looked[text].sum()), 3)) for text in examples})

In [ ]:
# 그림 11-11
fig, ax = plt.subplots(figsize=(7, 6.6))
board = [("로지스틱 회귀", "글자 1-그램", [baseline[1]], GRAY), ("로지스틱 회귀", "글자 1~2-그램", [baseline[2]], GRAY),
         ("로지스틱 회귀", "글자 1~3-그램", [baseline[3]], "#6b7480"),
         ("임베딩 평균", "어절", scores["임베딩 평균, 어절"], BRICK),
         ("임베딩 평균", "글자", scores["임베딩 평균, 글자"], NAVY),
         ("임베딩 평균", "부분 단어", scores["임베딩 평균, 부분 단어"], TEAL),
         ("어텐션으로 모으기", "글자", scores["어텐션으로 모으기, 글자"], NAVY),
         ("LSTM", "글자", scores["LSTM, 글자"], NAVY)]
heights = [0, 1, 2, 3.6, 4.6, 5.6, 6.6, 7.6]
for row, (model, unit, marks, shade) in zip(heights, board):
    middle = float(np.median(marks))
    ax.barh(row, middle, color=shade, height=0.64)
    if len(marks) > 1:
        ax.scatter(marks, [row] * len(marks), s=18, color="white", edgecolor="#222", lw=0.7, zorder=5)
        ax.text(87.3, row, f"{middle - baseline[3]:+.2f}%p".replace("-", "$-$"), va="center", fontsize=9, color=BRICK)
    ax.text(max(marks) + 0.3, row, f"{middle:.2f}%", va="center", fontsize=10.5, color="#111", weight="bold")
    ax.text(71.75, row - 0.13, model, va="center", ha="right", fontsize=10, color="#111", weight="bold")
    ax.text(71.75, row + 0.24, unit, va="center", ha="right", fontsize=9, color="#5b6470")
ax.axvline(baseline[3], color="#333", lw=1.0, ls="--")
ax.text(baseline[3] + 0.15, 2.82, f"넘어야 할 기준선 {baseline[3]:.2f}%", fontsize=9, color="#333", ha="right", va="center", bbox=BOX)
ax.text(87.3, 2.82, "기준선과의 차이", fontsize=9, color=BRICK, va="center")
ax.axhline(2.82, color="#c9ced6", lw=0.8, xmax=0.42)
ax.text(72.2, -0.72, "신경망을 쓰지 않는 기준선", fontsize=9, color="#5b6470", weight="bold")
ax.text(72.2, 2.98, "신경망 (시드 5개)", fontsize=9, color=NAVY, weight="bold")
ax.set_yticks([]); ax.set_ylim(8.2, -1.05); ax.set_xlim(72, 90.2)
ax.set_xticks(range(72, 88, 2))
ax.set_xlabel("검증 문장 10,000개의 정확도 (%). 막대는 시드 5개의 중앙값, 흰 점은 시드 하나의 값", fontsize=9)
frame(ax, grid="x")
ax.set_title("학습 문장 30,000개로 학습한 모델의 검증 정확도 (가로축은 72%에서 시작)", fontsize=10, loc="left", color=NAVY,
             weight="bold", x=-0.3)
ax.tick_params(axis="x", labelsize=9)
fig.subplots_adjust(left=0.25, right=0.97, top=0.94, bottom=0.09)
fig.savefig("fig_11_11.png", dpi=300)
plt.show()
print([(f"{model}, {unit}", round(float(np.median(marks)), 2)) for model, unit, marks, _ in board])